# Multi-Shield Quick Start

This notebook demonstrates how to use Multi-Shield to evaluate robust image classifiers against adversarial attacks.

## Overview

Multi-Shield combines a DNN classifier with a CLIP vision-language model to detect adversarial examples. When the two models disagree on a prediction, Multi-Shield rejects the sample.

In [1]:
import torch
from torchvision import transforms
from functools import partial

# Multi-Shield imports
from models.MultiShield import MultiShield
from models.CLIP import ClipModel
from ingredients.models import get_clip_model, get_local_model
from ingredients.dataset import get_dataset_loaders, get_label_names
from ingredients.utilities import set_seed, resize_to_224

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

/mnt/data2/fvillani/miniconda3/envs/multishield/lib/python3.11/site-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cuda


## 1. Setup

First, we'll set up the components:
- Load a robust DNN classifier from RobustBench
- Configure a CLIP model for zero-shot classification
- Create the Multi-Shield defense

In [2]:
# Configuration
SEED = 1233
DATASET = 'cifar10'
MODEL_NAME = 'carmon2019'  # Robust model from RobustBench
CLIP_MODEL_ID = 'tangake_finetuned'  # CLIP model fine-tuned on CIFAR-10
N_SAMPLES = 50  # Small number for quick demo
BATCH_SIZE = 16

# Set random seed for reproducibility
set_seed(SEED)

In [3]:
# Image normalization for CLIP
images_normalize = transforms.Normalize(
    (0.48145466, 0.4578275, 0.40821073), 
    (0.26862954, 0.26130258, 0.27577711)
)

# Get class labels
label_names = get_label_names(DATASET)
n_classes = len(label_names)
rejection_class = n_classes  # Rejection is an additional class
print(f'Classes: {label_names}')
print(f'Rejection class index: {rejection_class}')

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Rejection class index: 10


In [4]:
# Load dataset
dataloaders = get_dataset_loaders(DATASET, BATCH_SIZE, N_SAMPLES, SEED)
print(f'Loaded {len(dataloaders["val"].dataset)} samples')

Loading CIFAR10 dataset with batch size 16
whole length of the validation set is: 10000
Loaded 50 samples


In [5]:
# Load DNN classifier from RobustBench
print('Loading DNN classifier...')
dnn_model = get_local_model(MODEL_NAME, DATASET, images_normalize)
dnn_model = dnn_model.eval().to(device)
print(f'Loaded model: {MODEL_NAME}')

Loading DNN classifier...
Loading carmon2019
Loaded model: carmon2019


In [6]:
# Load CLIP model
print('Loading CLIP model...')
clip_model_name, processor_name, tokenizer_name, use_open_clip = get_clip_model(CLIP_MODEL_ID)

clip_model = ClipModel(
    clip_model_name,
    processor_name,
    tokenizer_name,
    use_open_clip,
    label_names,
    torch_preprocess=images_normalize,
    dataset=DATASET,
    device=device,
    resize=partial(resize_to_224) if DATASET in ['mnist', 'cifar10'] else None
)
print(f'Loaded CLIP model: {CLIP_MODEL_ID}')

Loading CLIP model...
Loaded CLIP model: tangake_finetuned


In [7]:
# Create Multi-Shield
multi_shield = MultiShield(dnn=dnn_model, clip_model=clip_model)
print('Multi-Shield initialized!')

Multi-Shield initialized!


## 2. Evaluate on Clean Samples

Let's see how Multi-Shield performs on clean (non-adversarial) samples.

In [8]:
# Get a batch of samples
images, labels = next(iter(dataloaders['val']))
images = images.to(device)
labels = labels.to(device)

print(f'Batch shape: {images.shape}')
print(f'Labels: {labels[:5].tolist()}')

Batch shape: torch.Size([16, 3, 32, 32])
Labels: [3, 9, 3, 1, 0]


In [9]:
# Run Multi-Shield
with torch.no_grad():
    outputs = multi_shield(images)

# The last dimension is the rejection score
predictions = outputs[:, :-1].argmax(dim=1)
rejection_scores = outputs[:, -1]

# A sample is rejected if the predicted class is the rejection class
final_predictions = outputs.argmax(dim=1)
rejected = final_predictions == rejection_class

print(f'DNN predictions: {predictions[:10].tolist()}')
print(f'True labels:     {labels[:10].tolist()}')
print(f'Rejected:        {rejected[:10].tolist()}')

DNN predictions: [3, 9, 3, 1, 0, 7, 0, 5, 8, 2]
True labels:     [3, 9, 3, 1, 0, 7, 0, 5, 8, 2]
Rejected:        [False, False, False, False, False, False, False, False, False, False]


In [10]:
# Compute clean accuracy and rejection rate
correct = (predictions == labels).sum().item()
n_rejected = rejected.sum().item()
n_total = len(labels)

print(f'\nClean Accuracy: {correct/n_total:.2%}')
print(f'Rejection Rate: {n_rejected/n_total:.2%}')


Clean Accuracy: 93.75%
Rejection Rate: 6.25%


## 3. Run Adversarial Attack

Now let's generate adversarial examples and see how Multi-Shield defends against them.

In [11]:
from attacks.modified_autoattack import AutoAttack

# Configure attack
epsilon = 8 / 255  # L-inf perturbation budget

# Non-adaptive attack (targets DNN only)
adversary = AutoAttack(
    dnn_model,
    rejection_class_index=None,
    norm='Linf',
    eps=epsilon,
    version='custom',
    attacks_to_run=['apgd-ce', 'apgd-dlr'],
    verbose=True,
    device=device
)
adversary.apgd.n_restarts = 1

In [12]:
# Generate adversarial examples
print('Generating adversarial examples...')
adv_images = adversary.run_standard_evaluation(images, labels)

Generating adversarial examples...
using custom version including apgd-ce, apgd-dlr.
initial accuracy: 93.75%
apgd-ce - 1/1 - 2 out of 15 successfully perturbed
robust accuracy after APGD-CE: 81.25% (total time 0.9 s)
apgd-dlr - 1/1 - 0 out of 13 successfully perturbed
robust accuracy after APGD-DLR: 81.25% (total time 1.7 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 81.25%


In [13]:
# Evaluate DNN on adversarial examples
with torch.no_grad():
    dnn_outputs = dnn_model(adv_images)
    dnn_adv_preds = dnn_outputs.argmax(dim=1)

dnn_correct = (dnn_adv_preds == labels).sum().item()
print(f'DNN Robust Accuracy: {dnn_correct/n_total:.2%}')

DNN Robust Accuracy: 81.25%


In [14]:
# Evaluate Multi-Shield on adversarial examples
with torch.no_grad():
    ms_outputs = multi_shield(adv_images)

ms_predictions = ms_outputs[:, :-1].argmax(dim=1)
ms_final = ms_outputs.argmax(dim=1)
ms_rejected = ms_final == rejection_class

# Accuracy: correct predictions + rejections (adversarial detected)
ms_correct = ((ms_predictions == labels) | ms_rejected).sum().item()
ms_rejection_rate = ms_rejected.sum().item()

print(f'Multi-Shield Robust Accuracy: {ms_correct/n_total:.2%}')
print(f'Multi-Shield Rejection Rate:  {ms_rejection_rate/n_total:.2%}')

Multi-Shield Robust Accuracy: 100.00%
Multi-Shield Rejection Rate:  18.75%


## Results Summary

Compare the performance of the standalone DNN vs Multi-Shield:

In [18]:
# Create results table
print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f'{"Metric":<40} {"Value":>15}')
print('-'*60)
print(f'{"Clean Accuracy (DNN)":<40} {correct/n_total:>14.2%}')
print(f'{"Clean Rejection Rate (MS)":<40} {n_rejected/n_total:>14.2%}')
print('-'*60)
print(f'{"Robust Accuracy (DNN)":<40} {dnn_correct/n_total:>14.2%}')
print(f'{"Robust Accuracy (Multi-Shield)":<40} {ms_correct/n_total:>14.2%}')
print(f'{"Adversarial Rejection Rate (MS)":<40} {ms_rejection_rate/n_total:>14.2%}')
print('-'*60)
print(f'{"Improvement (MS vs DNN)":<40} {(ms_correct - dnn_correct)/n_total:>14.2%}')
print('='*60)

# Additional statistics
print(f'\nTotal samples: {n_total}')
print(f'DNN fooled: {n_total - dnn_correct} samples')
print(f'Multi-Shield rejected: {ms_rejection_rate} samples')
print(f'Multi-Shield correctly classified: {(ms_predictions == labels).sum()} samples')


RESULTS SUMMARY
Metric                                             Value
------------------------------------------------------------
Clean Accuracy (DNN)                             93.75%
Clean Rejection Rate (MS)                         6.25%
------------------------------------------------------------
Robust Accuracy (DNN)                            81.25%
Robust Accuracy (Multi-Shield)                  100.00%
Adversarial Rejection Rate (MS)                  18.75%
------------------------------------------------------------
Improvement (MS vs DNN)                          18.75%

Total samples: 16
DNN fooled: 3 samples
Multi-Shield rejected: 3 samples
Multi-Shield correctly classified: 13 samples


## Summary

Multi-Shield improves robustness by rejecting samples where the DNN and CLIP disagree. When the DNN is fooled by an adversarial example but CLIP is not, Multi-Shield detects this disagreement and rejects the sample.

For a complete evaluation including adaptive attacks, use:
```bash
python main.py --config=configs/config_cifar10.json --device=cuda --seed=1233
```